# Fashion Recommendation & Classification System

**BSDC-DEVDP-28A · Introduction to Artificial Intelligence**
Team: Mara, Rayan

This notebook builds a fashion classification and recommendation system using the [Fashion Product Images (Small)](https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small) dataset (44,424 products). It covers data cleaning, exploratory analysis, three supervised classifiers, an evaluation of *why* accuracy plateaus around 77%, and a similarity-based recommender with an outfit builder extension.

## 1. Load the Data

In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv("/content/styles.csv", on_bad_lines='skip')

# Display the first 5 rows
df.head()

## 2. Explore the Data

We answer the core structural questions about the dataset to inform the rest of the analysis. Because the dataset contains a few malformed rows (text formatting glitches in product descriptions), we skip bad rows so the loader doesn't crash.

In [ ]:
# 1. Total products and columns
print("Dataset Shape:", df.shape)

# 2. Check for missing values
print("\n--- Missing Values ---")
print(df.isnull().sum())

# 3. See the most common colors
print("\n--- Top 5 Base Colours ---")
print(df['baseColour'].value_counts().head(5))

# 4. See the most common item types (what we want to predict)
print("\n--- Top 5 Article Types ---")
print(df['articleType'].value_counts().head(5))

**Key insights discovered**

- **Dataset size:** 44,424 total products and 10 attributes (gender, articleType, baseColour, season, etc.) — enough data to train a machine learning model.
- **Target feature (article types):** T-shirts are the most dominant item type (7,067 items), followed by Shirts (3,217) and Casual Shoes (2,845).
- **Colour distribution:** Black (9,728 items) and White (5,538 items) are the most frequent base colours.
- **Data flaws identified:** a small number of missing values — `usage` (317), `season` (21), `baseColour` (15), `productDisplayName` (7).

## 3. Clean the Data

In [ ]:
# 1. Drop rows where critical catalog information is missing
df_clean = df.dropna(subset=['baseColour', 'season', 'year', 'usage', 'productDisplayName'])

# 2. Verify the new size of our dataset
print("Original Dataset Rows:", df.shape[0])
print("Cleaned Dataset Rows:", df_clean.shape[0])
print("Total Rows Removed:", df.shape[0] - df_clean.shape[0])

# 3. Double-check that all missing values are gone
print("\n--- Remaining Missing Values ---")
print(df_clean.isnull().sum())

## 4. Feature Selection & Encoding

In [ ]:
# 1. Define our inputs (X) and target label (y)
features = ["gender", "masterCategory", "subCategory", "baseColour", "season", "usage"]
target = "articleType"

X = df_clean[features]
y = df_clean[target]

# 2. Convert text features into binary numbers (One-Hot Encoding)
X_encoded = pd.get_dummies(X)

print("--- Data Encoding Complete ---")
print("Original feature columns:", X.shape[1])
print("Encoded feature columns (0s and 1s):", X_encoded.shape[1])
X_encoded.head(3)

One-Hot Encoding turns every unique text category into its own column of 1s (yes) and 0s (no), so a categorical feature like `baseColour` becomes many binary columns the model can learn from.

## 5. Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.20, random_state=42)

print("--- Data Split Complete ---")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

`X_train` / `y_train` hold 80% of the data, used to train the model. `X_test` / `y_test` hold the remaining 20%, hidden from the model until evaluation. `random_state=42` makes the split reproducible.

## 6. Decision Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1. Initialize the Decision Tree model
clf_tree = DecisionTreeClassifier(random_state=42)

# 2. Train the model using the training set
clf_tree.fit(X_train, y_train)

# 3. Predict categories for the hidden test set
y_pred = clf_tree.predict(X_test)

# 4. Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print("--- Model Training Complete ---")
print(f"Decision Tree Baseline Accuracy: {accuracy * 100:.2f}%")

Out of 8,816 completely hidden fashion items, the Decision Tree correctly guessed the exact `articleType` over 3 out of 4 times, using only gender, colour, subCategory, season, and usage.

## 7. Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Initialize a Random Forest with 100 individual trees
clf_forest = RandomForestClassifier(n_estimators=100, random_state=42)

# 2. Train the forest
clf_forest.fit(X_train, y_train)

# 3. Predict on the hidden test set
y_pred_forest = clf_forest.predict(X_test)

# 4. Calculate accuracy
accuracy_forest = accuracy_score(y_test, y_pred_forest)
print("--- Random Forest Training Complete ---")
print(f"Random Forest Accuracy: {accuracy_forest * 100:.2f}%")
print(f"Improvement over Decision Tree: {(accuracy_forest - accuracy) * 100:.2f}%")

**How a Random Forest works**

A Random Forest is a committee of independent Decision Trees voting on the final answer, rather than trusting a single tree's judgement.

A single tree can overfit — get hyper-focused on a coincidental pattern in the training data. A Random Forest prevents this with two sources of randomness:

- **Random rows (bootstrapping):** each of the 100 trees sees only a random sample of the training rows, not all of them.
- **Random columns (feature subsampling):** when a tree decides how to split the data, it only considers a random subset of the 115 encoded columns at a time, not all of them.

Because every tree trains on a slightly different slice of data and features, they each develop a slightly different "opinion." When predicting, all 100 trees vote, and the majority prediction wins — e.g. if 78 trees vote "T-shirt" and 22 vote something else, the forest outputs "T-shirt." Averaging over 100 diverse trees cancels out the individual biases of any one tree, which is why Random Forest is generally more robust than a single Decision Tree.

## 8. Classification Reports

In [ ]:
from sklearn.metrics import classification_report

print("================ DECISION TREE REPORT ================")
print(classification_report(y_test, y_pred, zero_division=0))

print("\n================ RANDOM FOREST REPORT ================")
print(classification_report(y_test, y_pred_forest, zero_division=0))

## 9. K-Nearest Neighbors (Third Classifier)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# k=5 means: look at the 5 most similar items and take a majority vote
clf_knn = KNeighborsClassifier(n_neighbors=5)
clf_knn.fit(X_train, y_train)

y_pred_knn = clf_knn.predict(X_test)
accuracy_knn = accuracy_score(y_test, y_pred_knn)

print(f"KNN (k=5) Accuracy: {accuracy_knn * 100:.2f}%")

## 10. Model Comparison

In [ ]:
results = pd.DataFrame({
    "Algorithm": ["Decision Tree", "K-Nearest Neighbors", "Random Forest"],
    "Accuracy": [accuracy, accuracy_knn, accuracy_forest]
})
results["Accuracy"] = (results["Accuracy"] * 100).round(2).astype(str) + "%"
print(results.to_string(index=False))

## 11. Why Do All Three Models Land Within ~3 Points of Each Other?

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# How much of the accuracy comes from subCategory alone?
tests = {
    "All 6 features":                 ["gender","masterCategory","subCategory","baseColour","season","usage"],
    "subCategory ONLY":               ["subCategory"],
    "All features MINUS subCategory": ["gender","masterCategory","baseColour","season","usage"],
}

for name, feats in tests.items():
    X_t = pd.get_dummies(df_clean[feats])
    Xtr, Xte, ytr, yte = train_test_split(X_t, y, test_size=0.20, random_state=42)
    m = DecisionTreeClassifier(random_state=42).fit(Xtr, ytr)
    print(f"{name:35s} -> {accuracy_score(yte, m.predict(Xte)) * 100:.2f}%")

**Why Random Forest barely improved on the Decision Tree**

`masterCategory → subCategory → articleType` is a hierarchy, not three independent features. In 20 of the 45 subCategories, knowing the subCategory determines the articleType with complete certainty (e.g. Watches → Watches, Eyewear → Sunglasses).

The experiment above isolates this: `subCategory` alone reaches 56.22% accuracy. Every other feature combined, *without* subCategory, reaches only 54.56%. Together they reach 76.50%. So subCategory carries the model, and colour, season, gender, and usage contribute roughly twenty points on top.

This sets a ceiling on accuracy that no algorithm can exceed. Inside a subCategory such as Topwear, our features cannot distinguish a T-shirt from a Shirt — both are typically Men's, Black, Summer, Casual. The information simply isn't present in the data. This explains why Random Forest gained only +0.22% over a single Decision Tree, and why all three algorithms land within three percentage points of each other: **the limitation is the dataset's structure, not the choice of model.**

## 12. Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Focus on the 10 most common article types so the plot is readable
top10 = df_clean['articleType'].value_counts().head(10).index.tolist()

# Filter test set to only rows where the true label is in the top 10
mask = y_test.isin(top10)
y_test_top = y_test[mask]
y_pred_top = pd.Series(y_pred_forest, index=y_test.index)[mask]

cm = confusion_matrix(y_test_top, y_pred_top, labels=top10)

plt.figure(figsize=(9, 7))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xticks(range(len(top10)), top10, rotation=45, ha='right')
plt.yticks(range(len(top10)), top10)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Random Forest Confusion Matrix (Top 10 Article Types)')

for i in range(len(top10)):
    for j in range(len(top10)):
        plt.text(j, i, cm[i, j], ha='center', va='center',
                  color='white' if cm[i, j] > cm.max()/2 else 'black')

plt.tight_layout()
plt.show()

The confusion matrix confirms our earlier finding: most misclassifications happen between Tshirts and Shirts, since these two categories share nearly identical attributes (gender, colour, season, usage) in our data. This is a direct visual illustration of the subCategory ceiling described above.

## 13. Recommender System

In [ ]:
from sklearn.neighbors import NearestNeighbors

recommender = NearestNeighbors(n_neighbors=5, metric='euclidean')
recommender.fit(X_encoded)

def recommend(gender, masterCategory, subCategory, baseColour, season, usage, n=5):
    """
    Given user preferences, return the n most similar items in the catalogue.
    This finds neighbors - it does not predict a single label like the
    classifiers above.
    """
    user_input = pd.DataFrame([{
        "gender": gender, "masterCategory": masterCategory,
        "subCategory": subCategory, "baseColour": baseColour,
        "season": season, "usage": usage
    }])
    user_encoded = pd.get_dummies(user_input).reindex(columns=X_encoded.columns, fill_value=0)

    distances, indices = recommender.kneighbors(user_encoded, n_neighbors=n)
    results = df_clean.iloc[indices[0]][['productDisplayName', 'articleType', 'baseColour', 'season', 'usage']]
    return results

# Example: a casual black summer top for men
recommend(gender="Men", masterCategory="Apparel", subCategory="Topwear",
          baseColour="Black", season="Summer", usage="Casual")

## 14. Summary

- Three classifiers (Decision Tree, KNN, Random Forest) all converge around 74–77% accuracy because `subCategory` alone carries most of the predictive signal — a structural ceiling in the data, not a modelling weakness.
- The confusion matrix confirms this: nearly all errors happen between visually/functionally similar categories (Tshirts vs. Shirts).
- A separate `NearestNeighbors` recommender returns a ranked list of similar products given user preferences, fulfilling the project's recommendation-system goal alongside the classification analysis.
- A Streamlit demo application (`app.py`, in the accompanying GitHub repository) wraps this recommender in an interactive interface, including an Outfit Builder extension that assembles a complete top/bottom/shoes outfit filtered by season correctness.